#1. Basic Tasks 

##1. Load the messy e-commerce dataset and identify columns containing nulls and duplicate rows using .filter(), .distinct(), and .dropDuplicates(). 

In [0]:
import pyspark.sql.functions as F
df=spark.read.csv('/Volumes/dev/demo/raw/sales.csv',inferSchema=True,header=True)
df.display()
cols_with_nulls = [x for x in df.columns if df.filter(F.col(x).isNull()).count() > 0]
print(f"\ncolumns with atleast one value is null:{cols_with_nulls}")
# Identify duplicate rows
total_rows = df.count()
distinct_rows = df.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"\nTotal rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("\nSample of duplicate records:")
    df.groupBy(df.columns).count().filter(F.col("count") > 1).display()

# Drop duplicates
df_cleaned = df.dropDuplicates()
print(f"\nRows after removing duplicates: {df_cleaned.count()}")

##2. Rename at least 3 columns to a consistent naming convention (e.g., snake_case) using withColumnRenamed. 

In [0]:
df_renamed = df_cleaned \
    .withColumnRenamed("transaction_id", "transaction_code") \
    .withColumnRenamed("customer_id", "customer_code") \
    .withColumnRenamed("product_id", "product_code")

print("Original columns:", df_cleaned.columns)
print("Renamed columns:", df_renamed.columns)
df_renamed.display()

##3. Connect a Databricks Repo to a Git provider and make your first commit of a cleaning notebook. 


### Git Commands Used:
```bash
git add "Day 5 Assignment: PySpark DataFrame Transformations & Git Integration.ipynb"

# Commit with descriptive message
git commit -m "Add data cleaning notebook with PySpark transformations"

# Push to remote 
git push origin main
```

##4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove exact duplicates, and sort by order_date. 

In [0]:
# Improved cleaning pipeline with better null handling
df_sales = spark.read.csv('/Volumes/dev/demo/raw/sales.csv', inferSchema=True, header=True)
print(f"Initial rows: {df_sales.count()}")

# Step 1: Drop rows where ALL values are null (completely empty rows)
df_sales = df_sales.dropna(how='all')
print(f"After dropping fully-null rows: {df_sales.count()}")

# Step 2: Fill remaining nulls with sensible defaults
df_sales = df_sales.fillna({
    'quantity': 0,
    'discount_amount': 0.0,
    'total_amount': 0.0
})

# Step 3: Remove exact duplicates
df_sales = df_sales.dropDuplicates()
print(f"After removing duplicates: {df_sales.count()}")

# Step 4: Sort by order_date for chronological analysis
df_sales = df_sales.orderBy('order_date')

print("\nCleaned dataset:")
df_sales.display()

##5. Perform an aggregation (revenue by category or region) and a join against a second small reference table (e.g., customers or regions). 

In [0]:
from pyspark.sql import functions as f
df_cust=spark.read.json('/Volumes/cyntexa_dev/sales/external_data/orders.json',multiLine=True)
df_cust=df_cust.select(f.col("address.city").alias("city"), f.col("address.state").alias("state"),f.col("customer_id"),f.col("name"), f.explode('orders').alias("orders"))
df_cust.display()

In [0]:
df_cust_sales=df_cust.join(df_sales,'customer_id')

df_revenue_by_region = df_cust_sales.groupBy("state").agg(
    f.round(f.sum("total_amount"),2).alias("total_revenue"),
    f.count("transaction_id").alias("transaction_count")
).orderBy(f.desc("total_revenue"))

print("\nRevenue by Region (State):")
df_revenue_by_region.display()

##6. Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request describing what changed and why. 

## Feature Branch: Improved Cleaning Logic

### Branch: `feature/improve-cleaning-logic`

### Changes Made:

#### **Before (Original Logic):**
```python
df_sales = df_sales.dropna()  # Drops ANY row with ANY null
df_sales = df_sales.fillna({...})  # Never executes (no nulls left)
```

#### **After (Improved Logic):**
```python
# 1. Drop only fully-null rows (where ALL values are null)
df_sales = df_sales.dropna(how='all')

# 2. Fill remaining nulls with sensible defaults
df_sales = df_sales.fillna({
    'quantity': 0,
    'discount_amount': 0.0,
    'total_amount': 0.0
})

# 3. Remove exact duplicates
df_sales = df_sales.dropDuplicates()

# 4. Sort chronologically
df_sales = df_sales.orderBy('order_date')
```

### Why These Changes?

1. **Better Data Retention**: Original logic dropped rows with ANY null (lost 50 valid records). New logic preserves rows with partial data.

2. **Effective Null Handling**: Original fillna() never executed because dropna() removed all nulls first. New logic fills nulls BEFORE deduplication.

3. **Clear Pipeline Steps**: Each step now has a clear purpose and logs progress.

4. **Data Quality Metrics**: Added print statements to track data quality at each step:
   - Initial: 78 rows
   - After dropping fully-null rows: 70 rows (-8)
   - After deduplication: 50 rows (-20 duplicates)

### Impact:
* Retained 20 additional valid records that had partial null values
* Proper null imputation now functional
* Better audit trail with step-by-step counts